# Aviation Accidents Analysis

You are part of a consulting firm tasked to analyze commercial and passenger jet airline safety. The client (an airline/airplane insurer) wants to know what types of aircraft exhibit **low rates of total destruction** and **low likelihood of fatal or serious passenger injuries** in the event of an accident.

This notebook covers **Part 1: Data Loading, Inspection, and Cleaning**.

### Make Relevant Library Imports

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)


## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- Inspect NaNs, datatypes, and summary statistics

In [4]:
# Load the dataset
df = pd.read_csv('data/AviationData.csv', encoding='latin-1', low_memory=False)

print("Shape:", df.shape)
print()
df.head(3)


Shape: (88889, 31)



,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,Injury.Severity,Aircraft.damage,Aircraft.Category,Registration.Number,Make,Model,Amateur.Built,Number.of.Engines,Engine.Type,FAR.Description,Schedule,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
0,20001218X45444,Accident,SEA87LA080,1948-10-24,"MOOSE CREEK, ID",United States,NaN,NaN,NaN,NaN,Fatal(2),Destroyed,NaN,NC6404,Stinson,108-3,No,1.0,Reciprocating,NaN,NaN,Personal,NaN,2.0,0.0,0.0,0.0,UNK,Cruise,Probable Cause,NaN
1,20001218X45447,Accident,LAX94LA336,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,NaN,NaN,Fatal(4),Destroyed,NaN,N5069P,Piper,PA24-180,No,1.0,Reciprocating,NaN,NaN,Personal,NaN,4.0,0.0,0.0,0.0,UNK,Unknown,Probable Cause,19-09-1996
2,20061025X01555,Accident,NYC07LA005,1974-08-30,"Saltville, VA",United States,36.922223,-81.878056,NaN,NaN,Fatal(3),Destroyed,NaN,N5142R,Cessna,172M,No,1.0,Reciprocating,NaN,NaN,Personal,NaN,3.0,NaN,NaN,NaN,IMC,Cruise,Probable Cause,26-02-2007


In [5]:
# Inspect data types
df.dtypes


Event.Id                      str
Investigation.Type            str
Accident.Number               str
Event.Date                    str
Location                      str
Country                       str
Latitude                      str
Longitude                     str
Airport.Code                  str
Airport.Name                  str
Injury.Severity               str
Aircraft.damage               str
Aircraft.Category             str
Registration.Number           str
Make                          str
Model                         str
Amateur.Built                 str
Number.of.Engines         float64
Engine.Type                   str
FAR.Description               str
Schedule                      str
Purpose.of.flight             str
Air.carrier                   str
Total.Fatal.Injuries      float64
Total.Serious.Injuries    float64
Total.Minor.Injuries      float64
Total.Uninjured           float64
Weather.Condition             str
Broad.phase.of.flight         str
Report.Status 

In [6]:
# Summary statistics for numerical columns
df.describe()


,Number.of.Engines,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
count,82805.000000,77488.000000,76379.000000,76956.000000,82977.000000
mean,1.146585,0.647855,0.279881,0.357061,5.325440
std,0.446510,5.485960,1.544084,2.235625,27.913634
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,0.000000,1.000000
75%,1.000000,0.000000,0.000000,0.000000,2.000000
max,8.000000,349.000000,161.000000,380.000000,699.000000


In [7]:
# Check missing values as percentages
missing = (df.isna().mean() * 100).round(2).sort_values(ascending=False)
print("Missing values (%):")
print(missing)


Missing values (%):
Schedule                  85.85
Air.carrier               81.27
FAR.Description           63.97
Aircraft.Category         63.68
Longitude                 61.33
Latitude                  61.32
Airport.Code              43.60
Airport.Name              40.71
Broad.phase.of.flight     30.56
Publication.Date          15.49
Total.Serious.Injuries    14.07
Total.Minor.Injuries      13.42
Total.Fatal.Injuries      12.83
Engine.Type                7.98
Report.Status              7.18
Purpose.of.flight          6.97
Number.of.Engines          6.84
Total.Uninjured            6.65
Weather.Condition          5.05
Aircraft.damage            3.59
Registration.Number        1.55
Injury.Severity            1.12
Country                    0.25
Amateur.Built              0.11
Model                      0.10
Make                       0.07
Location                   0.06
Accident.Number            0.00
Investigation.Type         0.00
Event.Id                   0.00
Event.Date          

## Data Cleaning

### Filtering Aircraft and Events

We want to filter the dataset to include aircraft that the client is interested in:
- **Professional builds only** (Amateur.Built == 'No')
- **Airplanes only** (not helicopters, gliders, etc.) — where Aircraft.Category is 'Airplane' or NaN (many older records lack this field but are still airplanes)
- **Events from 1983 onwards** (max 40-year model lifetime)
- **Accidents only** (not incidents)

In [8]:
# Inspect relevant columns before filtering
print("Investigation.Type:")
print(df['Investigation.Type'].value_counts(dropna=False))
print()
print("Amateur.Built:")
print(df['Amateur.Built'].value_counts(dropna=False))
print()
print("Aircraft.Category:")
print(df['Aircraft.Category'].value_counts(dropna=False))


Investigation.Type:
Investigation.Type
Accident    85015
Incident     3874
Name: count, dtype: int64

Amateur.Built:
Amateur.Built
No     80312
Yes     8475
NaN      102
Name: count, dtype: int64

Aircraft.Category:
Aircraft.Category
NaN                  56602
Airplane             27617
Helicopter            3440
Glider                 508
Balloon                231
Gyrocraft              173
Weight-Shift           161
Powered Parachute       91
Ultralight              30
Unknown                 14
WSFT                     9
Powered-Lift             5
Blimp                    4
UNK                      2
Rocket                   1
ULTR                     1
Name: count, dtype: int64


In [9]:
# Convert Event.Date to datetime
df['Event.Date'] = pd.to_datetime(df['Event.Date'], errors='coerce')

# Filter: accidents only (not incidents)
df = df[df['Investigation.Type'].str.strip().str.lower() == 'accident'].copy()

# Filter: professional builds only
df = df[df['Amateur.Built'].str.strip().str.upper() == 'NO'].copy()

# Filter: airplanes only — keep rows where Aircraft.Category is 'Airplane' OR NaN
# (many older/general records lack category but are still airplanes)
df = df[(df['Aircraft.Category'].isna()) | (df['Aircraft.Category'].str.strip() == 'Airplane')].copy()

# Filter: 1983 onwards (40-year max model lifetime)
df = df[df['Event.Date'].dt.year >= 1983].copy()

print("Shape after filtering:", df.shape)


Shape after filtering: (69422, 31)


### Cleaning and Constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. We'll clean and impute relevant columns and then create derived fields.

**Construct metric for fatal/serious injuries**

Estimate the total number of passengers on each flight. The likelihood of serious/fatal injury is estimated as a fraction from this.

**Cleaning assumptions:**
- Injury counts (`Total.Fatal.Injuries`, `Total.Serious.Injuries`, `Total.Minor.Injuries`, `Total.Uninjured`) with NaN are imputed to 0, as missing counts likely indicate no injuries were recorded (i.e., zero) rather than truly unknown values — especially when other injury columns are filled.
- Total onboard = fatal + serious + minor + uninjured.
- `serious_fatal_fraction` = (fatal + serious) / total_onboard. If total_onboard == 0 (e.g., unmanned aircraft or data issue), set fraction to NaN to avoid division by zero.

In [10]:
# Inspect injury columns
injury_cols = ['Total.Fatal.Injuries', 'Total.Serious.Injuries',
               'Total.Minor.Injuries', 'Total.Uninjured']
print(df[injury_cols].isna().sum())
print()
df[injury_cols].describe()


Total.Fatal.Injuries       9053
Total.Serious.Injuries    10022
Total.Minor.Injuries       9553
Total.Uninjured            4602
dtype: int64



,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
count,60369.000000,59400.000000,59869.000000,64820.000000
mean,0.716030,0.288283,0.370860,3.688969
std,6.109423,1.658719,1.905966,21.628452
min,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,1.000000
75%,0.000000,0.000000,0.000000,2.000000
max,349.000000,161.000000,200.000000,699.000000


In [11]:
# Impute NaN injury counts to 0
df[injury_cols] = df[injury_cols].fillna(0)

# Compute total onboard
df['total_onboard'] = (df['Total.Fatal.Injuries'] + df['Total.Serious.Injuries']
                       + df['Total.Minor.Injuries'] + df['Total.Uninjured'])

# Compute serious/fatal fraction
df['serious_fatal_fraction'] = np.where(
    df['total_onboard'] > 0,
    (df['Total.Fatal.Injuries'] + df['Total.Serious.Injuries']) / df['total_onboard'],
    np.nan
)

print("serious_fatal_fraction stats:")
print(df['serious_fatal_fraction'].describe())
print()
print("Rows with total_onboard == 0:", (df['total_onboard'] == 0).sum())


serious_fatal_fraction stats:
count    69051.000000
mean         0.281787
std          0.432751
min          0.000000
25%          0.000000
50%          0.000000
75%          1.000000
max          1.000000
Name: serious_fatal_fraction, dtype: float64

Rows with total_onboard == 0: 371


**Aircraft.Damage**
- Identify and execute cleaning tasks
- Construct a derived column tracking whether an aircraft was destroyed or not.

In [12]:
# Inspect Aircraft.damage
print(df['Aircraft.damage'].value_counts(dropna=False))


Aircraft.damage
Substantial    52511
Destroyed      14952
NaN             1285
Minor            599
Unknown           75
Name: count, dtype: int64


In [13]:
# Standardize casing and strip whitespace
df['Aircraft.damage'] = df['Aircraft.damage'].str.strip().str.title()

# Create binary 'is_destroyed' column: 1 if Destroyed, 0 otherwise, NaN if unknown
df['is_destroyed'] = np.where(
    df['Aircraft.damage'].isna() | (df['Aircraft.damage'] == 'Unknown'),
    np.nan,
    (df['Aircraft.damage'] == 'Destroyed').astype(int)
)

print(df['Aircraft.damage'].value_counts(dropna=False))
print()
print("is_destroyed:")
print(df['is_destroyed'].value_counts(dropna=False))


Aircraft.damage
Substantial    52511
Destroyed      14952
NaN             1285
Minor            599
Unknown           75
Name: count, dtype: int64

is_destroyed:
is_destroyed
0.0    53110
1.0    14952
NaN     1360
Name: count, dtype: int64


### Investigate the *Make* Column
- Identify cleaning tasks
- Execute cleaning tasks
- Keep Makes with at least 50 records for statistical robustness

In [14]:
# Inspect Make — note many variations in case and spacing
print(df['Make'].value_counts().head(30))


Make
Cessna               20590
Piper                11138
CESSNA                4820
Beech                 3910
PIPER                 2797
Bell                  1743
Mooney                1012
BEECH                 1007
Grumman                974
Boeing                 862
Bellanca               816
Hughes                 680
Robinson               656
Air Tractor            575
Schweizer              542
Aeronca                457
BOEING                 444
Maule                  425
Champion               412
Aero Commander         335
De Havilland           329
Stinson                327
Rockwell               311
Taylorcraft            304
Luscombe               295
Mcdonnell Douglas      294
North American         287
Hiller                 279
Aerospatiale           263
MOONEY                 237
Name: count, dtype: int64


In [15]:
# Cleaning tasks for Make:
# 1. Strip whitespace
# 2. Title-case for consistency (e.g., 'CESSNA' -> 'Cessna', 'cessna' -> 'Cessna')
# 3. Fix known common aliases/misspellings

df['Make'] = df['Make'].str.strip().str.title()

# Manual correction of known aliases
make_corrections = {
    'Mcdonnell Douglas': 'McDonnell Douglas',
    'Mc Donnell Douglas': 'McDonnell Douglas',
    'Beechcraft': 'Beech',
    'Rockwell Commander': 'Rockwell',
    'Rockwell International': 'Rockwell',
    'De Havilland': 'De Havilland',
    'Dehavilland': 'De Havilland',
    'Douglas Aircraft': 'Douglas',
    'Socata': 'Socata',
    'Grumman American': 'Grumman',
    'Grumman Aerospace': 'Grumman',
    'Grumman Corp': 'Grumman',
    'Embraer S.A.': 'Embraer',
    'Embraer Sa': 'Embraer',
}
df['Make'] = df['Make'].replace(make_corrections)

# Keep Makes with >= 50 records
make_counts = df['Make'].value_counts()
valid_makes = make_counts[make_counts >= 50].index
df = df[df['Make'].isin(valid_makes)].copy()

print("Remaining unique makes:", df['Make'].nunique())
print("Shape:", df.shape)
print()
print(df['Make'].value_counts().head(20))


Remaining unique makes: 76
Shape: (64054, 34)

Make
Cessna            25410
Piper             13935
Beech              4942
Bell               1756
Boeing             1306
Grumman            1261
Mooney             1249
Bellanca            973
Hughes              681
Robinson            674
Air Tractor         671
Aeronca             606
Maule               569
Schweizer           547
Champion            502
De Havilland        461
Stinson             418
Rockwell            407
Aero Commander      404
Luscombe            390
Name: count, dtype: int64


### Inspect Model Column
- Remove NaNs
- Check if model labels are unique to each make
- Create a derived `make_model` column as a unique identifier for each aircraft type

In [16]:
# Inspect Model
print("Model NaN count:", df['Model'].isna().sum())
print("Sample models:", df['Model'].value_counts().head(20))


Model NaN count: 15
Sample models: Model
152          2215
172          1640
172N         1091
PA-28-140     862
172M          754
150           745
172P          661
182           611
180           595
PA-18         550
PA-18-150     549
150M          549
PA-28-180     546
PA-28-161     519
PA-28-181     505
150L          433
A36           426
G-164A        422
PA-38-112     421
140           385
Name: count, dtype: int64


In [17]:
# Drop rows with missing Model
df = df.dropna(subset=['Model']).copy()

# Strip and title-case Model
df['Model'] = df['Model'].str.strip().str.upper()

# Check if model names are unique to a make — do any models appear across different makes?
model_make_counts = df.groupby('Model')['Make'].nunique()
multi_make_models = model_make_counts[model_make_counts > 1]
print(f"Models appearing in more than one Make: {len(multi_make_models)}")
print(multi_make_models.head(10))


Models appearing in more than one Make: 342
Model
100        3
100-180    2
109        2
112        3
112A       2
112TCA     2
120        2
140        2
164        2
164B       2
Name: Make, dtype: int64


In [18]:
# Create a make_model unique identifier
df['make_model'] = df['Make'] + ' ' + df['Model']
print("Total unique make_model combinations:", df['make_model'].nunique())
print()
print(df['make_model'].value_counts().head(20))


Total unique make_model combinations: 5542

make_model
Cessna 152         2215
Cessna 172         1640
Cessna 172N        1091
Piper PA-28-140     862
Cessna 172M         754
Cessna 150          745
Cessna 172P         661
Cessna 182          611
Cessna 180          594
Piper PA-18         550
Piper PA-18-150     549
Cessna 150M         549
Piper PA-28-180     546
Piper PA-28-161     519
Piper PA-28-181     505
Cessna 150L         433
Piper PA-38-112     421
Beech A36           406
Cessna 140          384
Cessna 170B         374
Name: count, dtype: int64


### Cleaning Other Columns

Inspect and clean the following columns that may be related to accident outcomes:
- `Engine.Type`
- `Weather.Condition`
- `Number.of.Engines`
- `Purpose.of.flight`
- `Broad.phase.of.flight`

In [19]:
# Engine.Type
print("Engine.Type:")
print(df['Engine.Type'].value_counts(dropna=False))


Engine.Type:
Engine.Type
Reciprocating    53343
NaN               3474
Turbo Prop        2450
Turbo Shaft       2019
Unknown           1344
Turbo Fan         1102
Turbo Jet          306
UNK                  1
Name: count, dtype: int64


In [20]:
# Standardize Engine.Type — unify 'Unknown' variants, strip whitespace
df['Engine.Type'] = df['Engine.Type'].str.strip()
df['Engine.Type'] = df['Engine.Type'].replace({'Unknown': np.nan, 'UNK': np.nan, 'LR': np.nan,
                                                'NONE': np.nan, 'Hybrid Rocket': np.nan})
print(df['Engine.Type'].value_counts(dropna=False))


Engine.Type
Reciprocating    53343
NaN               4819
Turbo Prop        2450
Turbo Shaft       2019
Turbo Fan         1102
Turbo Jet          306
Name: count, dtype: int64


In [21]:
# Weather.Condition
print("Weather.Condition:")
print(df['Weather.Condition'].value_counts(dropna=False))


Weather.Condition:
Weather.Condition
VMC    56315
IMC     4940
NaN     2025
UNK      607
Unk      152
Name: count, dtype: int64


In [22]:
# Standardize Weather.Condition
df['Weather.Condition'] = df['Weather.Condition'].str.strip()
df['Weather.Condition'] = df['Weather.Condition'].replace({'UNK': np.nan, 'Unk': np.nan})
print(df['Weather.Condition'].value_counts(dropna=False))


Weather.Condition
VMC    56315
IMC     4940
NaN     2784
Name: count, dtype: int64


In [23]:
# Number.of.Engines
print("Number.of.Engines:")
print(df['Number.of.Engines'].value_counts(dropna=False))


Number.of.Engines:
Number.of.Engines
1.0    52050
2.0     7872
NaN     3242
0.0      501
4.0      199
3.0      175
Name: count, dtype: int64


In [24]:
# Number.of.Engines — keep as float (NaN remains NaN); cap unreasonable values
# Values above 8 are extremely rare and likely data errors
df.loc[df['Number.of.Engines'] > 8, 'Number.of.Engines'] = np.nan
print(df['Number.of.Engines'].value_counts(dropna=False))


Number.of.Engines
1.0    52050
2.0     7872
NaN     3242
0.0      501
4.0      199
3.0      175
Name: count, dtype: int64


In [25]:
# Purpose.of.flight
print("Purpose.of.flight:")
print(df['Purpose.of.flight'].value_counts(dropna=False))


Purpose.of.flight:
Purpose.of.flight
Personal                     35997
Instructional                 8766
Unknown                       4224
Aerial Application            3893
Business                      3220
NaN                           3076
Positioning                   1218
Other Work Use                 885
Public Aircraft                608
Ferry                          605
Aerial Observation             577
Executive/corporate            373
Skydiving                      174
Flight Test                    123
Banner Tow                      97
Public Aircraft - Federal       47
Glider Tow                      34
Public Aircraft - State         29
Firefighting                    20
Air Race/show                   17
Public Aircraft - Local         16
Air Race show                   16
External Load                   12
Air Drop                         6
PUBS                             3
ASHO                             3
Name: count, dtype: int64


In [26]:
# Standardize Purpose.of.flight — unify Unknown variants
df['Purpose.of.flight'] = df['Purpose.of.flight'].str.strip()
df['Purpose.of.flight'] = df['Purpose.of.flight'].replace({'Unknown': np.nan})
print(df['Purpose.of.flight'].value_counts(dropna=False))


Purpose.of.flight
Personal                     35997
Instructional                 8766
NaN                           7300
Aerial Application            3893
Business                      3220
Positioning                   1218
Other Work Use                 885
Public Aircraft                608
Ferry                          605
Aerial Observation             577
Executive/corporate            373
Skydiving                      174
Flight Test                    123
Banner Tow                      97
Public Aircraft - Federal       47
Glider Tow                      34
Public Aircraft - State         29
Firefighting                    20
Air Race/show                   17
Public Aircraft - Local         16
Air Race show                   16
External Load                   12
Air Drop                         6
PUBS                             3
ASHO                             3
Name: count, dtype: int64


In [27]:
# Broad.phase.of.flight
print("Broad.phase.of.flight:")
print(df['Broad.phase.of.flight'].value_counts(dropna=False))


Broad.phase.of.flight:
Broad.phase.of.flight
NaN            16254
Landing        12539
Takeoff         9409
Cruise          8013
Maneuvering     6094
Approach        4956
Taxi            1494
Climb           1490
Descent         1443
Go-around       1169
Standing         722
Unknown          384
Other             72
Name: count, dtype: int64


In [28]:
# Standardize Broad.phase.of.flight
df['Broad.phase.of.flight'] = df['Broad.phase.of.flight'].str.strip()
df['Broad.phase.of.flight'] = df['Broad.phase.of.flight'].replace({'Unknown': np.nan})
print(df['Broad.phase.of.flight'].value_counts(dropna=False))


Broad.phase.of.flight
NaN            16638
Landing        12539
Takeoff         9409
Cruise          8013
Maneuvering     6094
Approach        4956
Taxi            1494
Climb           1490
Descent         1443
Go-around       1169
Standing         722
Other             72
Name: count, dtype: int64


### Column Removal
- Inspect the dataframe and drop columns that have too many NaNs or are not useful for analysis

In [29]:
# Inspect NaN percentages for all columns
missing_pct = (df.isna().mean() * 100).round(1).sort_values(ascending=False)
print(missing_pct)


Schedule                  87.2
Air.carrier               84.7
FAR.Description           73.2
Aircraft.Category         73.1
Longitude                 64.8
Latitude                  64.8
Airport.Code              42.3
Airport.Name              39.6
Broad.phase.of.flight     26.0
Publication.Date          17.7
Purpose.of.flight         11.4
Engine.Type                7.5
Number.of.Engines          5.1
Report.Status              5.1
Weather.Condition          4.3
is_destroyed               1.8
Aircraft.damage            1.7
Registration.Number        1.3
serious_fatal_fraction     0.5
Injury.Severity            0.3
Country                    0.3
Location                   0.1
Event.Id                   0.0
Accident.Number            0.0
Investigation.Type         0.0
Make                       0.0
Event.Date                 0.0
Model                      0.0
Amateur.Built              0.0
Total.Serious.Injuries     0.0
Total.Fatal.Injuries       0.0
Total.Uninjured            0.0
Total.Mi

In [30]:
# Drop columns with more than 60% missing or that are not relevant to the analysis
cols_to_drop = ['Latitude', 'Longitude', 'Airport.Code', 'Airport.Name',
                'Schedule', 'Air.carrier', 'FAR.Description', 'Report.Status',
                'Publication.Date', 'Event.Id', 'Accident.Number', 'Registration.Number']

df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
print("Shape after column removal:", df.shape)
print()
print("Remaining columns:")
print(df.columns.tolist())


Shape after column removal: (64039, 23)

Remaining columns:
['Investigation.Type', 'Event.Date', 'Location', 'Country', 'Injury.Severity', 'Aircraft.damage', 'Aircraft.Category', 'Make', 'Model', 'Amateur.Built', 'Number.of.Engines', 'Engine.Type', 'Purpose.of.flight', 'Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured', 'Weather.Condition', 'Broad.phase.of.flight', 'total_onboard', 'serious_fatal_fraction', 'is_destroyed', 'make_model']


In [31]:
# Final check on cleaned dataframe
print("Final shape:", df.shape)
print()
df.head(5)


Final shape: (64039, 23)



,Investigation.Type,Event.Date,Location,Country,Injury.Severity,Aircraft.damage,Aircraft.Category,Make,Model,Amateur.Built,Number.of.Engines,Engine.Type,Purpose.of.flight,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,total_onboard,serious_fatal_fraction,is_destroyed,make_model
3601,Accident,1983-01-01,"NEWPORT, OR",United States,Non-Fatal,Substantial,NaN,Cessna,182P,No,1.0,Reciprocating,Personal,0.0,0.0,1.0,3.0,VMC,Approach,4.0,0.0,0.0,Cessna 182P
3602,Accident,1983-01-01,"WOODBINE, IA",United States,Non-Fatal,Substantial,NaN,Cessna,182RG,No,1.0,Reciprocating,Personal,0.0,0.0,0.0,2.0,VMC,Landing,2.0,0.0,0.0,Cessna 182RG
3603,Accident,1983-01-01,"MARYVILLE, MO",United States,Non-Fatal,Substantial,NaN,Cessna,182P,No,1.0,Reciprocating,Personal,0.0,0.0,0.0,1.0,VMC,Takeoff,1.0,0.0,0.0,Cessna 182P
3604,Accident,1983-01-01,"UPLAND, CA",United States,Non-Fatal,Substantial,NaN,Piper,PA-28R-200,No,1.0,Reciprocating,Personal,0.0,0.0,2.0,0.0,VMC,Approach,2.0,0.0,0.0,Piper PA-28R-200
3605,Accident,1983-01-01,"SPRINGBROOK, WI",United States,Non-Fatal,Substantial,NaN,Cessna,140,No,1.0,Reciprocating,Instructional,0.0,0.0,0.0,2.0,VMC,Landing,2.0,0.0,0.0,Cessna 140


### Save DataFrame to CSV

Save the cleaned data for use in the analysis notebook.

In [32]:
# Save cleaned dataframe
df.to_csv('data/aviation_cleaned.csv', index=False)
print("Cleaned data saved to data/aviation_cleaned.csv")
print("Shape:", df.shape)


Cleaned data saved to data/aviation_cleaned.csv
Shape: (64039, 23)
